## Import Libs

In [8]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Import for get the environment variables 
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")

## Configurations from Spark

In [5]:
conf = SparkConf()

conf.setAppName("Delta Table History") # Name of spark application, useful for logs
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", f"{MINIO_USER}") # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", f"{MINIO_PASSWORD}") # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True)
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

## Create Data Frame

In [6]:
exampleData = [("Rodrigo", "Maturino", "M", 4000),
            ("Rafael", "Souza", "M", 4500),
            ("Luigi", "Bispo", "M", 6000),
            ("Eduarda", "Santos", "F", 5000)]

schema = StructType([
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("salary", IntegerType(), True)
        ])

dataFrame = spark.createDataFrame(data=exampleData, schema=schema)
dataFrame.show()

+----------+---------+------+------+
|first_name|last_name|Gender|salary|
+----------+---------+------+------+
|   Rodrigo| Maturino|     M|  4000|
|    Rafael|    Souza|     M|  4500|
|     Luigi|    Bispo|     M|  6000|
|   Eduarda|   Santos|     F|  5000|
+----------+---------+------+------+



In [10]:
table_path_data_feed = "s3a://bronze/table_data_feed"

## Enable the change Data Feed

In [11]:
dataFrame.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed", "true").save(table_path_data_feed)

## Verify Change Data Feed. The Data Feed logs all changes to the table: INSERT, UPDATE, and DELETE

## adding new colunms like:

### _change_type
### insert
### update_preimage
### update_postimage
### delete
### _commit_version
### _commit_timestamp


In [12]:
delta_table = DeltaTable.forPath(spark, table_path_data_feed)
details_dataFrame = delta_table.detail()

## The "explode()" is used to “denormalize” columns containing arrays or maps, converting each element into a new row.

In [17]:
details_dataFrame = details_dataFrame.select(explode(col("properties")).alias("key", "value"))
dataFeed_verify = details_dataFrame.filter(col("key") == "delta.enableChangeDataFeed").select("value").show()

AnalysisException: Column 'properties' does not exist. Did you mean one of the following? [key, value];
'Project [explode('properties) AS Buffer(key, value)]
+- Project [key#886, value#887]
   +- Generate explode(properties#831), false, [key#886, value#887]
      +- CommandResult [format#821, id#822, name#823, description#824, location#825, createdAt#826, lastModified#827, partitionColumns#828, numFiles#829L, sizeInBytes#830L, properties#831, minReaderVersion#832, minWriterVersion#833], Execute DescribeDeltaDetailCommand, [[delta,22546dda-0ac6-4986-892d-cadce4dde238,null,null,s3a://bronze/table_data_feed,1777058424778000,1777058431000000,[],4,4851,keys: [delta.enableChangeDataFeed], values: [true],1,4]]
            +- DescribeDeltaDetailCommand s3a://bronze/table_data_feed


## How to read the change data feed
## The "startingVersion" specifies which version of the Delta table you want to start reading the changes from

In [18]:
df_with_dataFeed = spark.read.format("delta").option("readChangeData", "true").option("startingVersion", 0).load(table_path_data_feed)
df_with_dataFeed.show()

+----------+---------+------+------+------------+---------------+--------------------+
|first_name|last_name|Gender|salary|_change_type|_commit_version|   _commit_timestamp|
+----------+---------+------+------+------------+---------------+--------------------+
|   Rodrigo| Maturino|     M|  4000|      insert|              0|2026-04-24 19:20:...|
|   Eduarda|   Santos|     F|  5000|      insert|              0|2026-04-24 19:20:...|
|    Rafael|    Souza|     M|  4500|      insert|              0|2026-04-24 19:20:...|
|     Luigi|    Bispo|     M|  6000|      insert|              0|2026-04-24 19:20:...|
+----------+---------+------+------+------------+---------------+--------------------+

